# Week 2 — LangChain, LangGraph & Data Collection

**Week 2 · Class 2–3** — Prompt templates, LangGraph agent, scraping, PDF chunking

## Learning objectives

- Code **zero-shot, few-shot, chain-of-thought, and zero-shot CoT** with LangChain
- Deep dive on **message roles** and `MessagesPlaceholder`
- Build a **LangGraph** agent with classify → chat / study routing
- Scrape data from **quotes.toscrape.com** (stretch: books.toscrape.com)
- Parse a **PDF**, chunk text, and wire chunks into your agent

Get a free API key at [console.groq.com](https://console.groq.com) before Section 2.

> Run cells **top to bottom**. Code matches the Class 2–3 slides.



## Section 1 — Install packages

Install LangChain, LangGraph, Groq integration, and data-collection libraries.


In [ ]:
!pip install -q langchain langchain-core langchain-groq langgraph requests beautifulsoup4 pymupdf


## Section 2 — Groq API key

Use `getpass` in Colab — never commit API keys to Git.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))


## Section 3 — Zero-shot prompt

**Technique:** instruction only, no examples.

Exam prep: summarize photosynthesis for revision.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0.7)

zero_shot = ChatPromptTemplate.from_messages([
    ("system", "You are a Biology exam tutor."),
    ("human", (
        "Summarize the key concepts of {topic} I need for a {duration} exam. "
        "Use bullet points: definition, inputs/outputs, light vs dark reactions, one common exam trap."
    )),
])

chain = zero_shot | llm
response = chain.invoke({"topic": "photosynthesis", "duration": "45-minute"})
print(response.content)


## Section 4 — Few-shot prompt

**Technique:** show 1 example (mitosis flashcard), then ask for photosynthesis.

The `("ai", ...)` turn is the assistant role — the model mirrors this format.


In [ ]:
few_shot = ChatPromptTemplate.from_messages([
    ("system", "You create exam flashcards in a fixed format."),
    ("human", "Topic: Mitosis"),
    ("ai", (
        "- Term: Mitosis\n"
        "- Definition: Cell division producing two identical diploid cells\n"
        "- Exam tip: Don't confuse with meiosis (4 haploid cells)"
    )),
    ("human", "Topic: {topic}"),
])

chain = few_shot | llm
response = chain.invoke({"topic": "Photosynthesis"})
print(response.content)


## Section 5 — Chain-of-thought (CoT)

**Technique:** explicit step-by-step reasoning for worked problems.


In [ ]:
cot = ChatPromptTemplate.from_messages([
    ("system", "You are a Biology tutor. Always show your reasoning step by step."),
    ("human", "{problem}"),
])

problem = (
    "A plant fixes 12 molecules of CO₂ in one hour via the Calvin cycle. "
    "Walk through each stage (carbon fixation → reduction → regeneration). "
    "Show your reasoning at each step, then state how many G3P molecules are produced."
)

chain = cot | llm
response = chain.invoke({"problem": problem})
print(response.content)


## Section 6 — Zero-shot chain-of-thought

**Technique:** trigger phrase *"Let's think step by step"* — no examples needed.


In [ ]:
zs_cot = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful exam coach."),
    ("human", (
        "Let's think step by step.\n"
        "Compare {topic_a} and {topic_b}.\n"
        "Which process is more efficient at converting light energy to usable chemical energy, and why?\n"
        "End with three revision questions I should practice."
    )),
])

chain = zs_cot | llm
response = chain.invoke({
    "topic_a": "photosynthesis",
    "topic_b": "cellular respiration",
})
print(response.content)


## Section 7 — Roles deep dive

| LangChain tuple | API role | Use |
|-----------------|----------|-----|
| `("system", ...)` | system | Bot rules, tone, constraints |
| `("human", ...)` | user | Current question |
| `("ai", ...)` | assistant | Few-shot examples or history |
| `MessagesPlaceholder` | dynamic | Multi-turn memory slot |


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful exam tutor. Keep answers concise."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

# Example: one prior turn
history = [
    HumanMessage(content="What is ATP?"),
    AIMessage(content="ATP is the cell's energy currency — adenosine triphosphate."),
]

messages = chat_with_history.format_messages(
    history=history,
    input="How does ATP relate to photosynthesis?",
)
for m in messages:
    print(f"{m.type}: {m.content[:80]}...")


## Section 8 — When to put few-shot examples where

- **System message** — global rules ("always use flashcard format")
- **Assistant (`ai`) turns** — concrete format demonstrations (few-shot)
- **MessagesPlaceholder** — real conversation history (multi-turn)
- **Human message** — the actual task for this turn


In [ ]:
# Compare: few-shot in assistant turn vs. instructions-only system prompt
demo = ChatPromptTemplate.from_messages([
    ("system", "You are an exam coach."),
    ("human", "Define {term} in one sentence for exam revision."),
])

print(demo.format_messages(term="chlorophyll")[1].content)


## Section 9 — LangGraph: define state

A **StateGraph** holds shared data passed between nodes.


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    intent: str
    context_chunks: str


## Section 10 — LangGraph: classify node

Routes user input to `chat` (general Q&A) or `study` (flashcards / CoT).


In [ ]:
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Classify the user message as exactly one word: chat or study.\n"
        "- chat: general questions, definitions, casual Q&A\n"
        "- study: flashcards, exam prep, step-by-step problems, revision"
    )),
    ("human", "{user_input}"),
])

def classify_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    user_text = last.content if hasattr(last, "content") else str(last)
    result = (classify_prompt | llm).invoke({"user_input": user_text})
    intent = result.content.strip().lower()
    if "study" in intent:
        intent = "study"
    else:
        intent = "chat"
    return {"intent": intent}


## Section 11 — LangGraph: chat and study nodes


In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly tutor. Answer clearly in under 5 sentences."),
    ("human", "{user_input}"),
])

study_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an exam prep coach. Use few-shot flashcard format or step-by-step reasoning. "
        "If context is provided, base your answer on it."
    )),
    ("human", "Context:\n{context}\n\nTask: {user_input}"),
])

def chat_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    result = (chat_prompt | llm).invoke({"user_input": user_text})
    return {"messages": [AIMessage(content=result.content)]}

def study_node(state: AgentState) -> dict:
    user_text = state["messages"][-1].content
    context = state.get("context_chunks") or "(no extra context)"
    result = (study_prompt | llm).invoke({
        "user_input": user_text,
        "context": context,
    })
    return {"messages": [AIMessage(content=result.content)]}


## Section 12 — LangGraph: build the graph


In [ ]:
def route(state: AgentState) -> str:
    return state["intent"]

graph = StateGraph(AgentState)
graph.add_node("classify", classify_node)
graph.add_node("chat", chat_node)
graph.add_node("study", study_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"chat": "chat", "study": "study"})
graph.add_edge("chat", END)
graph.add_edge("study", END)

agent = graph.compile()
print("LangGraph agent compiled.")


## Section 13 — Test routing: chat vs study


In [ ]:
test_prompts = [
    "What is photosynthesis?",
    "Make me flashcards for photosynthesis",
    "Walk me through this Calvin cycle problem step by step",
]

for prompt in test_prompts:
    result = agent.invoke({
        "messages": [HumanMessage(content=prompt)],
        "intent": "",
        "context_chunks": "",
    })
    print(f"\n--- Input: {prompt}")
    print(f"Intent: {result['intent']}")
    print(f"Reply: {result['messages'][-1].content[:300]}...")


## Section 14 — Assignment checkpoint

Extend the graph if needed:
- Improve the classify prompt for your phrasing
- Add logging to see which node runs
- Test with your own exam topics

**No solution for the full extension is provided — you complete it in Section 25.**


In [ ]:
# Your turn: test one custom prompt
custom = "Create revision questions for the Krebs cycle"
result = agent.invoke({
    "messages": [HumanMessage(content=custom)],
    "intent": "",
    "context_chunks": "",
})
print("Intent:", result["intent"])
print(result["messages"][-1].content)


## Section 15 — Web scraping setup

We use [quotes.toscrape.com](https://quotes.toscrape.com) — a site built for scraping practice.


In [ ]:
import requests
from bs4 import BeautifulSoup

SCRAPE_URL = "https://quotes.toscrape.com/"
response = requests.get(SCRAPE_URL, timeout=10)
response.raise_for_status()
print("Status:", response.status_code)
soup = BeautifulSoup(response.text, "html.parser")


## Section 16 — Extract quotes


In [ ]:
quotes = []
for item in soup.select("div.quote"):
    text_el = item.select_one("span.text")
    author_el = item.select_one("small.author")
    tags = [t.get_text(strip=True) for t in item.select("a.tag")]
    if text_el and author_el:
        quotes.append({
            "text": text_el.get_text(strip=True),
            "author": author_el.get_text(strip=True),
            "tags": tags,
        })

print(f"Scraped {len(quotes)} quotes")
for q in quotes[:3]:
    print(f"  - {q['text'][:60]}... — {q['author']}")


## Section 17 — Scrape ethics reminder

- Check `robots.txt` before scraping any site
- Add `time.sleep(1)` between requests on multi-page scrapes
- Use practice sites or APIs in production


In [ ]:
import urllib.robotparser

rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://quotes.toscrape.com/robots.txt")
rp.read()
print("Can fetch /:", rp.can_fetch("*", SCRAPE_URL))


## Section 18 — Save scraped data


In [ ]:
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

quotes_path = data_dir / "scraped_quotes.json"
with open(quotes_path, "w", encoding="utf-8") as f:
    json.dump(quotes, f, indent=2, ensure_ascii=False)

print(f"Saved to {quotes_path}")


## Section 19 — Stretch: books.toscrape.com

Scrape book titles and prices from the first page (extend to pagination in Section 20).


In [ ]:
BOOKS_URL = "https://books.toscrape.com/"
books_resp = requests.get(BOOKS_URL, timeout=10)
books_resp.raise_for_status()
books_soup = BeautifulSoup(books_resp.text, "html.parser")

books = []
for article in books_soup.select("article.product_pod"):
    title = article.select_one("h3 a")
    price = article.select_one("p.price_color")
    if title and price:
        books.append({
            "title": title.get("title", title.get_text(strip=True)),
            "price": price.get_text(strip=True),
        })

print(f"Scraped {len(books)} books")
for b in books[:3]:
    print(f"  - {b['title']} — {b['price']}")


## Section 20 — Stretch: pagination

Follow `next` links to scrape more pages (cap at 3 pages to be polite).


In [ ]:
import time

def scrape_book_page(url: str) -> list[dict]:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    page_soup = BeautifulSoup(resp.text, "html.parser")
    page_books = []
    for article in page_soup.select("article.product_pod"):
        title = article.select_one("h3 a")
        price = article.select_one("p.price_color")
        if title and price:
            page_books.append({
                "title": title.get("title", title.get_text(strip=True)),
                "price": price.get_text(strip=True),
            })
    next_link = page_soup.select_one("li.next a")
    next_url = None
    if next_link and next_link.get("href"):
        next_url = "https://books.toscrape.com/" + next_link["href"].lstrip("/")
    return page_books, next_url

all_books = []
url = BOOKS_URL
for _ in range(3):
    page_books, url = scrape_book_page(url)
    all_books.extend(page_books)
    if not url:
        break
    time.sleep(1)

books_path = data_dir / "scraped_books.json"
with open(books_path, "w", encoding="utf-8") as f:
    json.dump(all_books, f, indent=2, ensure_ascii=False)
print(f"Saved {len(all_books)} books to {books_path}")


## Section 21 — PDF text extraction

Use the sample PDF in `week-2/data/sample.pdf` or upload your own course document.


In [ ]:
import fitz  # PyMuPDF

pdf_path = Path("data/sample.pdf")
if not pdf_path.exists():
    # Colab: download sample if running from repo root
    alt = Path("week-2/data/sample.pdf")
    pdf_path = alt if alt.exists() else pdf_path

if pdf_path.exists():
    doc = fitz.open(pdf_path)
    full_text = "".join(page.get_text() for page in doc)
    print(f"Extracted {len(full_text)} characters from {doc.page_count} pages")
    print(full_text[:400])
else:
    print("Upload a PDF to data/sample.pdf or set pdf_path to your file.")
    full_text = ""  # replace after uploading


## Section 22 — Alternative: pdfplumber

Useful for PDFs with tables.


In [ ]:
# !pip install -q pdfplumber
# import pdfplumber
# with pdfplumber.open("data/sample.pdf") as pdf:
#     full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
# print(full_text[:400])


## Section 23 — Chunking for embeddings


In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        start = end - overlap
    return chunks

chunks = chunk_text(full_text) if full_text else []
print(f"Created {len(chunks)} chunks")
if chunks:
    print("First chunk:", chunks[0][:200], "...")

chunks_path = data_dir / "chunks.json"
with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f"Saved to {chunks_path}")


## Section 24 — Wire chunks into LangGraph study node


In [ ]:
sample_chunk = chunks[0] if chunks else "Photosynthesis converts light energy into chemical energy in chloroplasts."

result = agent.invoke({
    "messages": [HumanMessage(content="Summarize this context for exam revision as bullet points")],
    "intent": "",
    "context_chunks": sample_chunk,
})
print("Intent:", result["intent"])
print(result["messages"][-1].content)


## Section 25 — Deliverable checklist

Complete these for your Week 2 lab (no solution code provided):

- [ ] LangGraph agent routes **chat** vs **study** for 3+ test prompts
- [ ] `data/scraped_quotes.json` saved (stretch: `scraped_books.json`)
- [ ] At least one PDF parsed and chunked to `data/chunks.json`
- [ ] Study node uses `context_chunks` from your dataset
- [ ] Demo ready: show routing + scrape sample + PDF chunk + one prompt template

**Week 3:** your chunks become embeddings in FAISS / Qdrant.


In [ ]:
# Final self-check — uncomment and run when done
# assert Path("data/scraped_quotes.json").exists(), "Missing scraped_quotes.json"
# assert Path("data/chunks.json").exists(), "Missing chunks.json"
# print("Deliverable files present. Ready for Week 3!")
